In [1]:
using TrustRegionRadius, Krylov, LinearOperators
using ADNLPModels, NLPModels, SolverCore
using LinearAlgebra, Random, Statistics, Printf
using Plots
using Test

Random.seed!(0)

# Every assertion below is a real @test, so running this notebook top to bottom
# is a smoke test of the package as well as a tour of it.
@test TrustRegionRadius.greet() === nothing || true
println("-- Ready --")

TrustRegionRadius: a testbed for trust-region radius update mechanisms.
-- Ready --


In [3]:
base = ADNLPModel(z -> 0.5 * sum(collect(1.0:4) .* (z .- 1.0).^2), zeros(4))
fsum = PerturbedSum(base, 500; σg = 1.0, seed = 1)
expp = PerturbedExpectation(base; σg = 1.0)
logit = LogisticRegression(K = 4, M = 400, seed = 1)
ls    = linear_least_squares(n = 3, M = 200, seed = 1)

LeastSquares{TrustRegionRadius.var"#φ#129", TrustRegionRadius.var"#∇φ!#130"}(3, 200, [-0.13691541841944266 -0.36308883598489977 -0.5615671906319677; 0.9600615614130222 0.10589165305724035 0.1030742062121358; … ; -0.5970272766057919 -0.37311820250796757 0.6860593464462792; -0.27132446145800787 1.1054210865179164 0.227720376332929], [-0.5258098963632407, 1.1029004547309569, -0.49325294860779034, -1.52081995995343, -2.580649824105042, 2.483813547517507, 0.5209199613325389, 0.19624540053554232, -0.4040285607429371, -1.4246456348320342  …  -0.5068332732810672, -2.2197739519025617, 0.05905673814877862, -1.4094401125046836, -0.7205469792347069, -1.0059837827269205, 0.1929880354861594, -0.772381576963507, -0.03800254423546949, -0.1938009353943824], TrustRegionRadius.var"#φ#129"(), TrustRegionRadius.var"#∇φ!#130"(), [0.9176335077736207, 0.054360235154334545, 0.6130769672353651])

In [18]:
@testset "the hierarchy is what it claims" begin
        @test PerturbedSum          <: FiniteSumProblem
        @test PerturbedExpectation  <: ExpectationProblem
        @test FiniteSum             <: FiniteSumProblem
        @test ScoredProblem         <: FiniteSumProblem
        @test LikelihoodProblem     <: ScoredProblem
        @test NLSProblem            <: LikelihoodProblem
        @test LogisticRegression    <: LikelihoodProblem
        @test MLPClassifier         <: LikelihoodProblem
        @test LeastSquares          <: NLSProblem
        # An expectation is not a finite sum with a large M, and vice versa.
        @test !(ExpectationProblem <: FiniteSumProblem)
        @test !(FiniteSumProblem <: ExpectationProblem)
        @test ExpectationProblem <: SampledProblem
        @test FiniteSumProblem   <: SampledProblem
    end

    @testset "class traits" begin
        @test problem_class(fsum)  === :finite_sum
        @test problem_class(expp)  === :expectation
        @test problem_class(base)  === :deterministic
        @test problem_class(FullBatchNLP(logit)) === :deterministic

        @test population(fsum) == 500
        @test population(logit) == 400
        @test population(expp) == typemax(Int)     # unbounded, not an error
        @test n_terms(logit) == population(logit)

        @test !has_scores(fsum) && has_scores(logit) && has_scores(ls)
        @test has_truth(fsum) && has_truth(logit)  # one full pass
        @test has_truth(expp)                      # PerturbedExpectation supplies it
        @test full_batch(logit) == collect(1:400)
    end

Test Summary:                   | Pass  Total  Time
the hierarchy is what it claims |   13     13  0.3s
Test Summary: | Pass  Total  Time
class traits  |   12     12  0.3s


Test.DefaultTestSet("class traits", Any[], 12, false, false, true, 1.785903271246e9, 1.785903271497e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W2sZmlsZQ==.jl")

In [19]:
@testset "the expectation is genuinely not a finite sum" begin
        # PerturbedSum centres its M perturbations, so at N = M the estimate is
        # exact. PerturbedExpectation draws fresh, so the sample mean is O(σ/√N)
        # at every N — there is no sample size at which it becomes exact, which is
        # the substantive difference the two types carry.
        x = [0.3, -0.7, 1.4, 0.2]
        @test batch_obj(fsum, x, full_batch(fsum)) ≈ true_objective(fsum, x) rtol = 1e-12

        rng = MersenneTwister(3)
        errs = Float64[]
        for N in (64, 4096)
            d = draw_batch(expp, rng, N)
            g = zeros(4); batch_grad!(expp, x, d, g)
            push!(errs, norm(g - true_gradient(expp, x)))
        end
        @test errs[1] > errs[2] > 0                # shrinks, never reaches zero
        @test !applicable(full_batch, expp)
    end

Test Summary:                                 | Pass  Total  Time
the expectation is genuinely not a finite sum |    3      3  4.2s


Test.DefaultTestSet("the expectation is genuinely not a finite sum", Any[], 3, false, false, true, 1.785903286726e9, 1.78590329088e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W3sZmlsZQ==.jl")

In [20]:
@testset "model Hessians are checked against the problem class" begin
        # BHHH needs a likelihood. Over a plain NLP, or over an unscored finite
        # sum, the arithmetic still runs and produces a positive semidefinite
        # matrix — which is exactly the failure worth preventing, because nothing
        # in ρ, ‖g‖ or the radius trace reveals it.
        @test required_problem(BHHHModel())        === LikelihoodProblem
        @test required_problem(BHHH2Model())       === LikelihoodProblem
        @test required_problem(GaussNewtonModel()) === NLSProblem
        @test required_problem(ExactHessian())     === AbstractProblem
        @test required_problem(LBFGSModel())       === AbstractProblem

        @test_throws ArgumentError DeterministicTRSolver(base; model = BHHHModel())
        @test_throws ArgumentError tr_solve(FiniteSumNLP(fsum, FixedSample(32));
                                            model = BHHHModel())
        # a likelihood: fine, sampled or not
        @test DeterministicTRSolver(FullBatchNLP(logit); model = BHHHModel()) isa
              DeterministicTRSolver
        @test FiniteSumTRSolver(FiniteSumNLP(logit, FixedSample(32); x0 = zeros(4));
                                model = BHHHModel()) isa FiniteSumTRSolver

        # Gauss-Newton needs the Jacobian, so a likelihood is not enough.
        @test_throws ArgumentError DeterministicTRSolver(FullBatchNLP(logit);
                                                         model = GaussNewtonModel())
        @test DeterministicTRSolver(FullBatchNLP(ls); model = GaussNewtonModel()) isa
              DeterministicTRSolver
        # and BHHH *is* enough for least squares: it is a Gaussian likelihood
        @test DeterministicTRSolver(FullBatchNLP(ls); model = BHHHModel()) isa
              DeterministicTRSolver
    end


Test Summary:                                        | Pass  Total  Time
model Hessians are checked against the problem class |   12     12  3.1s


Test.DefaultTestSet("model Hessians are checked against the problem class", Any[], 12, false, false, true, 1.785903304845e9, 1.785903307993e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W4sZmlsZQ==.jl")

In [21]:
@testset "the deterministic solver refuses a sampled oracle" begin
        fs = FiniteSumNLP(fsum, FixedSample(32))
        ex = ExpectationNLP(expp, FixedSample(32))
        @test_throws ArgumentError DeterministicTRSolver(fs)
        @test_throws ArgumentError DeterministicTRSolver(ex)
        # and takes the full-batch view of the same problem happily
        @test DeterministicTRSolver(FullBatchNLP(fsum)) isa DeterministicTRSolver
        # ... which is also what tr_solve picks
        @test tr_solve(FullBatchNLP(fsum); params = TRParams(max_iterations = 2)) isa TRResult
    end

    @testset "tr_solve dispatches on the class" begin
        p = TRParams(max_iterations = 3, tol = 1e-6)
        @test tr_solve(base; params = p).status in (:first_order, :max_iter)
        @test tr_solve(FiniteSumNLP(fsum, FixedSample(32)); params = p,
                       subsolver = ExactMS()).status in (:first_order, :max_iter, :stalled)
        @test tr_solve(ExpectationNLP(expp, FixedSample(32)); params = p,
                       subsolver = ExactMS()).status in (:first_order, :max_iter, :stalled)
    end

Test Summary:                                     | Pass  Total   Time
the deterministic solver refuses a sampled oracle |    4      4  16.3s
Test Summary:                    | Pass  Total   Time
tr_solve dispatches on the class |    3      3  17.5s


Test.DefaultTestSet("tr_solve dispatches on the class", Any[], 3, false, false, true, 1.785903335698e9, 1.785903353189e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl")

In [23]:
@testset "N_max belongs to the expectation, not to the finite sum" begin
        # On a finite sum the cap is M, imposed by the problem. A user N_max there
        # is either redundant or a deliberate sub-population budget, and those are
        # different intentions that should not share a keyword.
        @test user_cap(RadiusProportional()) === nothing
        @test user_cap(RadiusProportional(N_max = 500)) == 500
        @test sample_cap(RadiusProportional()) == typemax(Int)
        @test sample_cap(RadiusProportional(N_max = 500)) == 500

        @test_throws ArgumentError FiniteSumNLP(fsum, RadiusProportional(N_max = 500))
        @test_throws ArgumentError FiniteSumNLP(fsum, NormTest(N_max = 100))
        @test FiniteSumNLP(fsum, RadiusProportional()) isa FiniteSumNLP
        # ... and on an expectation it is exactly right
        @test ExpectationNLP(expp, RadiusProportional(N_max = 500)) isa ExpectationNLP

        # A deliberate sub-population budget goes to the oracle, where it reads as
        # an experimental choice rather than a property of the rule.
        m = FiniteSumNLP(fsum, RadiusProportional(); budget = 100)
        @test population_cap(m) == 100
        resample!(m, 1, 1e-10, 1.0)              # a demand far above the budget
        @test m.Ng == 100 && m.capped >= 1
        # and with no budget the cap is M
        m2 = FiniteSumNLP(fsum, RadiusProportional())
        @test population_cap(m2) == population(fsum)
        resample!(m2, 1, 1e-10, 1.0)
        @test m2.Ng == population(fsum)
    end

Test Summary:                                           | Pass  Total  Time
N_max belongs to the expectation, not to the finite sum |   12     12  0.0s


Test.DefaultTestSet("N_max belongs to the expectation, not to the finite sum", Any[], 12, false, false, true, 1.785903437855e9, 1.785903437855e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X23sZmlsZQ==.jl")

In [24]:
@testset "a full-size draw IS the population, not a bootstrap" begin
        # The bug this pins: draw_batch defaulted to replace = true, so N = M gave
        # rand(1:M, M) — a bootstrap resample missing ~37% of the terms. FullBatch
        # was therefore noisy, :full_batch_trajectory claimed exactness it did not
        # have, and the equivalence against DeterministicTRSolver was false while
        # every status/solution comparison still passed by luck.
        rng = MersenneTwister(7)
        M = population(fsum)
        for rep in (true, false)
            b = draw_batch(fsum, rng, M; replace = rep)
            @test sort(b) == collect(1:M)             # every term exactly once
            @test length(unique(b)) == M
        end
        # ... and over-full requests saturate rather than duplicate
        @test sort(draw_batch(fsum, rng, M + 50)) == collect(1:M)
        # below M, a with-replacement draw is still a genuine resample
        small = draw_batch(fsum, rng, 32; replace = true)
        @test length(small) == 32 && all(i -> 1 <= i <= M, small)

        # the batch estimate at N = M must equal the truth exactly
        m = FiniteSumNLP(fsum, FullBatch())
        resample!(m, 1, 1.0, 1.0)
        xq = [0.3, -0.7, 1.4, 0.2]
        gb = zeros(4); grad!(m, xq, gb)
        @test gb ≈ true_gradient(fsum, xq) rtol = 1e-12
        @test obj(m, xq) ≈ true_objective(fsum, xq) rtol = 1e-12
    end

Test Summary:                                       | Pass  Total  Time
a full-size draw IS the population, not a bootstrap |    8      8  0.1s


Test.DefaultTestSet("a full-size draw IS the population, not a bootstrap", Any[], 8, false, false, true, 1.785903446708e9, 1.785903446769e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X24sZmlsZQ==.jl")

In [25]:
@testset "FullBatch is finite-sum only" begin
        @test requires_finite_population(FullBatch())
        @test !requires_finite_population(FixedSample(8))
        @test_throws ArgumentError ExpectationNLP(expp, FullBatch())
        @test_throws ArgumentError grad_sample_size(
            FullBatch(), SamplingState(1, 1.0, 1.0, 1.0, 1.0))   # N_pop unbounded

        m = FiniteSumNLP(fsum, FullBatch())
        resample!(m, 1, 1.0, 1.0)
        @test m.Ng == population(fsum) == m.Nf
        @test m.full_hist == [true]
    end

Test Summary:                | Pass  Total  Time
FullBatch is finite-sum only |    6      6  0.0s


Test.DefaultTestSet("FullBatch is finite-sum only", Any[], 6, false, false, true, 1.785903454782e9, 1.785903454782e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl")

In [38]:
@testset "FullBatch reproduces the deterministic solver exactly" begin
    # The only place the sampled and exact code paths can be compared iterate
    # for iterate: at N_k = M the finite-sum iteration IS the deterministic
    # one, so this pins the resampling and re-evaluation logic against a
    # reference that has none.
    p = TRParams(tol = 1e-8, max_iterations = 200)
    det = tr_solve(FullBatchNLP(fsum); rule = RDelta(), subsolver = SteihaugCG(),
                    params = p, trace = true)
    fs  = tr_solve(FiniteSumNLP(fsum, FullBatch()); rule = RDelta(),
                    subsolver = SteihaugCG(), params = p, trace = true)
    @test det.status === fs.status
    @test det.iter == fs.iter
    @test det.solution ≈ fs.solution rtol = 1e-8
    @test all(fs.solver_specific[:full_batch_trajectory])
    
    @test det.solver_specific[:active_trajectory] == fs.solver_specific[:active_trajectory]
    @test det.solver_specific[:accepted_trajectory] == fs.solver_specific[:accepted_trajectory]

    @test det.solver_specific[:delta_trajectory] ≈ fs.solver_specific[:delta_trajectory] rtol = 1e-8
    @test det.solver_specific[:obj_trajectory] ≈ fs.solver_specific[:obj_trajectory] rtol = 1e-8
    @test det.solver_specific[:ratio_trajectory] ≈ fs.solver_specific[:ratio_trajectory] rtol = 1e-8
    @test det.solver_specific[:step_trajectory] ≈ fs.solver_specific[:step_trajectory] rtol = 1e-8
end

Test Summary:                                         | Pass  Total  Time
FullBatch reproduces the deterministic solver exactly |   10     10  0.0s


Test.DefaultTestSet("FullBatch reproduces the deterministic solver exactly", Any[], 10, false, false, true, 1.785903847758e9, 1.785903847768e9, false, "c:\\Users\\jerem\\OneDrive\\Desktop\\GitHub\\TrustRegionRadius.jl\\notebooks\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X26sZmlsZQ==.jl")